## **PORTAFOLIO ACADÉMICO DE BI:**
# **PROYECTO INMOBILIARIO — CIUDAD DE LA COSTA**
# **Geografía comparativa — Norte, Sur y Detalles**
---
## Módulo 3:
* Introducción
* Conexión y recuperación de Datos
* Preguntas empresariales:
1. Precio por barrio y zona
2. Perfil de Tipo de inmueble
3. Equipamiento
4. Volatilidad de precio
5. Premio de precio por proximidad

---
## **Introducción**
Con los módulos de infraestructura (1) y rentabilidad (2) cerrados, este módulo presenta comparativas en zona Norte y zona Sur dentro de Ciudad de la Costa usando la Av. Giannattasio como línea divisoria de referencia.

Buscamos validar con datos si existe una diferencia real de precio, tipología y equipamiento entre ambos lados.

**Recordatorio de Metadatos y Gobernanza:**
Todos los hallazgos de este Módulo siguen sujetos a los mismos límites metodológicos ya documentados por lo que cada conclusión de negocio debe leerse con ese contexto de fondo.

---

### **Conexión y recuperación de Datos del Módulo 1:**

Para dar inicio al **Módulo 3** debemos vincular el cuaderno de trabajo con la infraestructura de datos que ya dejamos consolidada.

In [ ]:
# Autenticamos la cuenta de Google en este nuevo cuaderno
from google.colab import auth
auth.authenticate_user()

print("Autenticación completa")

Autenticación completa


In [ ]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd

# 1. Nos autenticamos asegurando el proyecto destino
auth.authenticate_user(project_id="proyectosuy")

# 2. Inicializamos el cliente pasándole explícitamente el ID en el constructor
client = bigquery.Client(project="proyectosuy")

# 3. Definimos la consulta
query_toda_la_base = """
    SELECT *
    FROM `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
"""

# 4. Cargamos todo el DataFrame completo
# Pasamos el project_id también en el método de ejecución por seguridad
df_completo = client.query(query_toda_la_base, project="proyectosuy").to_dataframe()

# Verificamos que cargaron todas las columnas y filas
print(f"Base de datos cargada. Total de propiedades: {len(df_completo)}")
df_completo.head()

Base de datos cargada. Total de propiedades: 555


,id,publicacion,finalizacion,operacion,tipo_inmueble,moneda,precio,ubicacion,zona,pisos,...,banos,cochera,parrillero,jardin,piscina,mts2_terreno,mts2_edificado,antiguedad,gastos_comunes_UYU,detalles
0,1,2025-06-01,NaT,Venta,Casa,U$S,289000.0,Solymar,Sur,2,...,4,1,True,True,False,364,139.0,20,NaN,Construcción sólida
1,2,2026-05-07,NaT,Venta,Casa,U$S,250000.0,Solymar,Sur,1,...,2,1,True,True,False,310,95.0,1,NaN,Construcción sólida
2,3,2026-05-05,NaT,Venta,Complejo,U$S,187000.0,Solymar,Norte,2,...,2,2,True,False,False,148,78.2,0,NaN,Próximo a Car One
3,4,2026-01-12,NaT,Venta,Casa,U$S,265000.0,Solymar,Sur,1,...,2,1,True,True,False,527,210.0,30,NaN,Casa independiente
4,5,2026-04-02,NaT,Venta,Casa,U$S,185000.0,Lagomar,Norte,1,...,1,1,True,True,False,200,85.0,1,NaN,Próximo a Almenara Mall


# **Preguntas empresariales:**

### **Pregunta 1. Precio por barrio y zona**
Antes de comparar Norte y Sur de forma global necesitamos confirmar si esa comparación tiene sentido dentro de cada barrio o si algún barrio en particular distorsiona el promedio general de su zona.

Calculamos el precio de venta (tanto la media como la mediana) cruzando barrio y zona simultáneamente, en vez de solo zona.
#### **Nota metodológica:**
Al cruzar barrio y zona, algunas combinaciones pueden tener muestras chicas (o inexistentes si un barrio no tiene presencia real en una de las dos zonas) por lo que cualquier diferencia encontrada debe leerse junto con la **cantidad de propiedades** de cada grupo.


In [ ]:
# =============================================================================
# PRECIO (MEDIA Y MEDIANA) POR BARRIO Y ZONA
# =============================================================================

# Calculamos, para cada combinación de barrio + zona, el precio promedio
# (media) y la mediana de venta de viviendas construidas, en dólares.
query_precio_barrio_zona = """
WITH datos_individuales AS (
    SELECT
        ubicacion AS barrio,
        zona,
        tipo_inmueble,

        -- Precio de venta de vivienda (excluyendo terrenos), en USD.
        CASE WHEN LOWER(tipo_inmueble) != 'terreno'
              AND LOWER(moneda) = 'u$s'
              AND LOWER(operacion) = 'venta'
         THEN precio END AS precio_venta_vivienda,

        -- Mediana calculada por función de ventana, particionada por
        -- barrio + zona a la vez, para que cada combinación tenga su
        -- propia mediana en vez de una mediana global de la zona.
        PERCENTILE_CONT(
            CASE WHEN LOWER(tipo_inmueble) != 'terreno'
                  AND LOWER(moneda) = 'u$s'
                  AND LOWER(operacion) = 'venta'
             THEN precio END,
            0.5
        ) OVER (PARTITION BY ubicacion, zona) AS mediana_precio_ventana
    FROM
        `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
)

# Agregamos por barrio + zona para obtener la tabla final comparativa.
SELECT
    barrio,
    zona,

    -- Cantidad de viviendas consideradas en el cálculo (permite evaluar
    -- la confiabilidad de cada combinación barrio + zona).
    COUNTIF(precio_venta_vivienda IS NOT NULL) AS cantidad_viviendas,

    ROUND(AVG(precio_venta_vivienda), 0) AS media_precio_usd,
    ROUND(MAX(mediana_precio_ventana), 0) AS mediana_precio_usd,

    -- Brecha entre media y mediana como % de la mediana: cuanto más alta,
    -- más asimetría hay en los precios de esa combinación (outliers
    -- empujando el promedio hacia arriba o hacia abajo).
    ROUND(
        SAFE_DIVIDE(
            AVG(precio_venta_vivienda) - MAX(mediana_precio_ventana),
            MAX(mediana_precio_ventana)
        ) * 100, 1
    ) AS brecha_media_mediana_pct
FROM
    datos_individuales
GROUP BY
    barrio, zona
ORDER BY
    zona, media_precio_usd DESC;
"""

# Ejecutamos la consulta y la cargamos en el DataFrame de Pandas.
df_precio_barrio_zona = client.query(query_precio_barrio_zona).to_dataframe()

# Desplegamos el resultado.
df_precio_barrio_zona

,barrio,zona,cantidad_viviendas,media_precio_usd,mediana_precio_usd,brecha_media_mediana_pct
0,Tahona,Norte,30,650033.0,590000.0,10.2
1,Carrasco,Norte,9,296067.0,150000.0,97.4
2,Solymar,Norte,58,215981.0,208700.0,3.5
3,Shangrila,Norte,4,210000.0,220000.0,-4.5
4,Lagomar,Norte,9,197889.0,185000.0,7.0
5,Pinar,Norte,7,110286.0,120000.0,-8.1
6,Lagomar,Sur,16,348563.0,330000.0,5.6
7,Pinar,Sur,11,331455.0,272000.0,21.9
8,Carrasco,Sur,12,324500.0,295000.0,10.0
9,Shangrila,Sur,28,315286.0,315000.0,0.1


### **Diagnóstico: Precio por barrio y zona**
**1. El hallazgo más importante:** Carrasco norte tiene una brecha Media/Mediana del 97.4%, la más extrema de todo el proyecto hasta ahora. La media es de USD 296,067, pero la mediana es de apenas USD 150,000, la media casi duplica a la mediana. Esto es una señal clara de uno o dos outliers de precio muy alto dentro de una muestra ya chica (solo 9 viviendas) y confirma que los promedios "por zona" pueden estar escondiendo distorsiones fuertes a nivel de barrio. Este dato hay que leerlo con la mediana (USD 150,000), no con la media.
**2. La Tahona:** Ahora tenemos una brecha propia y documentada de 10.2% entre media y mediana (moderada) consistente con lo que ya sabíamos del Módulo 1 (el segmento premium con algunas mansiones empujando el promedio hacia arriba).
**3. El patrón "Sur más caro" se sostiene, pero ahora con matices por barrio:** Mirando la mediana (más confiable que la media): Lagomar sur (USD 330,000) es el más caro, seguido de Shangrilá sur (USD 315,000).

En el norte la mediana más alta es la de La Tahona (USD 590,000) que sigue siendo el máximo absoluto del Dataset (ahora sabemos que es un caso aislado dentro del norte, no representativo del resto de esa zona).

**4. Pinar norte y Shangrilá norte tienen brecha negativa (mediana por encima de la media: -8.1% y -4.5%):** En esas combinaciones específicas hay algunas propiedades más baratas de lo normal tirando el promedio hacia abajo, el patrón inverso al que veníamos viendo en el resto del proyecto.
**Advertencia de calidad de datos:** muestras muy chicas en el norte como
Shangrilá (4), Pinar (7), Carrasco (9) y Lagomar (9) tienen menos de 10 casos cada una, **cualquier lectura sobre estas combinaciones específicas es orientativa, no concluyente**.
La desagregación por barrio y zona nos da más precisión pero a cambio de reducir el tamaño de cada grupo.

---

### **Pregunta 2. Perfil de Tipo de inmueble**
Ya vimos que el precio varía entre combinaciones de barrio/zona, ahora buscamos una explicación estructural: ¿la composición de tipologías (terreno, casa, dúplex, etc) también varía dentro del mismo barrio según la zona?.

Esto nos permite entender si la brecha de precio responde a diferencias reales de "qué se vende" en cada zona más allá de la ubicación en sí.

In [ ]:
# =============================================================================
# PERFIL DE TIPO DE INMUEBLE POR BARRIO Y ZONA
# =============================================================================

# Contamos, para cada combinación de barrio + zona, cuántas propiedades
# hay de cada tipo de inmueble, y calculamos qué porcentaje representa
# cada tipo dentro de esa combinación específica (no de la zona global).
query_tipo_inmueble_barrio_zona = """
WITH conteo_barrio_zona_tipo AS (
    SELECT
        ubicacion AS barrio,
        zona,
        tipo_inmueble,
        COUNT(*) AS cantidad_propiedades
    FROM
        `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
    GROUP BY
        barrio, zona, tipo_inmueble
)

# Calculamos el porcentaje que representa cada tipo de inmueble DENTRO
# de su combinación barrio + zona, usando una función de ventana para
# el total de cada grupo.
SELECT
    barrio,
    zona,
    tipo_inmueble,
    cantidad_propiedades,
    SUM(cantidad_propiedades) OVER (PARTITION BY barrio, zona) AS total_propiedades_grupo,

    -- Porcentaje que representa este tipo de inmueble dentro de su
    -- combinación específica de barrio + zona.
    ROUND(
        SAFE_DIVIDE(
            cantidad_propiedades,
            SUM(cantidad_propiedades) OVER (PARTITION BY barrio, zona)
        ) * 100, 1
    ) AS porcentaje_del_grupo
FROM
    conteo_barrio_zona_tipo
ORDER BY
    barrio, zona, porcentaje_del_grupo DESC;
"""

# Ejecutamos la consulta y la cargamos en el DataFrame de Pandas.
df_tipo_inmueble_barrio_zona = client.query(query_tipo_inmueble_barrio_zona).to_dataframe()

# Desplegamos el resultado.
df_tipo_inmueble_barrio_zona

,barrio,zona,tipo_inmueble,cantidad_propiedades,total_propiedades_grupo,porcentaje_del_grupo
0,Carrasco,Norte,Terreno,20,30,66.7
1,Carrasco,Norte,Duplex,5,30,16.7
2,Carrasco,Norte,Apartamento,2,30,6.7
3,Carrasco,Norte,Complejo,2,30,6.7
4,Carrasco,Norte,Casa,1,30,3.3
5,Carrasco,Sur,Casa,8,14,57.1
6,Carrasco,Sur,Duplex,5,14,35.7
7,Carrasco,Sur,Complejo,1,14,7.1
8,Lagomar,Norte,Casa,5,13,38.5
9,Lagomar,Norte,Complejo,4,13,30.8


### **Diagnóstico: perfil de tipo de inmueble por barrio y zona**
**1. Carrasco norte es el barrio con mayor concentración de terreno de todo el Dataset.** Un 66.7% de las propiedades de Carrasco norte son terrenos sin construir (el porcentaje más alto de toda la tabla), esto explica directamente la brecha del 97.4% entre media y mediana que vimos en la Pregunta 1: con tan pocas viviendas reales en esa combinación (solo 9 de 30 propiedades son "vivienda" propiamente dicha) cualquier mansión aislada que se cuele en el cálculo de precio de "vivienda" tiene un efecto desproporcionado sobre el promedio.

**2. El patrón "norte = desarrollo sur = consolidado" se confirma barrio por barrio.** En todos los barrios donde el terreno tiene presencia relevante en el norte (Carrasco 66.7% Solymar 33.3% Pinar 21.1% Lagomar 23.1%) esa misma categoría cae drásticamente o desaparece en el Sur del mismo barrio (Carrasco 0% Solymar 4.2% Pinar 11.1% Lagomar 0%). Esto no es un patrón de "la zona en general", es un patrón que se repite de forma consistente dentro de cada barrio individual.

**3. Solymar es el barrio con la composición más completa y equilibrada en ambas zonas.** Tiene las 5 tipologías presentes tanto en el norte como en el sur con volúmenes grandes en ambos casos (147 y 167 propiedades respectivamente), es el barrio más representativo para sacar conclusiones generales sobre la dinámica norte/sur gracias a su volumen.

**4. La Tahona no tiene zona sur y su zona norte reproduce el patrón "desarrollo + consolidado".** Con 53.2% casas y 40.3% terreno, La Tahona combina ambos perfiles: es simultáneamente el segmento más premium ya construido y una oportunidad de desarrollo activa, sin apenas dúplex o complejos (3.2% cada uno) coherente con su perfil exclusivo de baja densidad.

---

### **Pregunta 3. Equipamiento**
Ya vimos que la composición de tipologías varía entre zonas dentro de cada barrio, ahora completamos el perfil con el nivel de confort de las viviendas ya construidas (cochera, parrillero, jardín, piscina).

In [ ]:
# =============================================================================
# EQUIPAMIENTO POR BARRIO Y ZONA
# =============================================================================

# Calculamos, para cada combinación de barrio + zona, el porcentaje de
# viviendas construidas (excluyendo terrenos) que cuentan con cada amenity.
query_equipamiento_barrio_zona = """
SELECT
    ubicacion AS barrio,
    zona,

    -- Cantidad de viviendas construidas consideradas en este cruce
    -- (excluimos terrenos porque no aplican a estas variables de confort).
    COUNTIF(LOWER(tipo_inmueble) != 'terreno') AS cantidad_viviendas,

    -- Porcentaje de viviendas con cochera, dentro de esta combinación.
    ROUND(
        SAFE_DIVIDE(
            COUNTIF(LOWER(tipo_inmueble) != 'terreno' AND cochera > 0),
            COUNTIF(LOWER(tipo_inmueble) != 'terreno')
        ) * 100, 1
    ) AS porcentaje_cochera,

    -- Porcentaje de viviendas con parrillero, dentro de esta combinación.
    ROUND(
        SAFE_DIVIDE(
            COUNTIF(LOWER(tipo_inmueble) != 'terreno' AND parrillero = TRUE),
            COUNTIF(LOWER(tipo_inmueble) != 'terreno')
        ) * 100, 1
    ) AS porcentaje_parrillero,

    -- Porcentaje de viviendas con jardín, dentro de esta combinación.
    ROUND(
        SAFE_DIVIDE(
            COUNTIF(LOWER(tipo_inmueble) != 'terreno' AND jardin = TRUE),
            COUNTIF(LOWER(tipo_inmueble) != 'terreno')
        ) * 100, 1
    ) AS porcentaje_jardin,

    -- Porcentaje de viviendas con piscina, dentro de esta combinación.
    -- Foco especial acá: buscamos confirmar si el "efecto La Tahona" es
    -- el único responsable de la piscina siendo más común en el Norte.
    ROUND(
        SAFE_DIVIDE(
            COUNTIF(LOWER(tipo_inmueble) != 'terreno' AND piscina = TRUE),
            COUNTIF(LOWER(tipo_inmueble) != 'terreno')
        ) * 100, 1
    ) AS porcentaje_piscina
FROM
    `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
GROUP BY
    barrio, zona
ORDER BY
    zona, porcentaje_piscina DESC;
"""

# Ejecutamos la consulta y la cargamos en el DataFrame de Pandas.
df_equipamiento_barrio_zona = client.query(query_equipamiento_barrio_zona).to_dataframe()

# Desplegamos el resultado.
df_equipamiento_barrio_zona

,barrio,zona,cantidad_viviendas,porcentaje_cochera,porcentaje_parrillero,porcentaje_jardin,porcentaje_piscina
0,Tahona,Norte,37,89.2,97.3,97.3,75.7
1,Carrasco,Norte,10,70.0,60.0,50.0,40.0
2,Solymar,Norte,98,84.7,69.4,92.9,7.1
3,Lagomar,Norte,10,100.0,90.0,100.0,0.0
4,Pinar,Norte,15,80.0,33.3,93.3,0.0
5,Shangrila,Norte,6,66.7,83.3,83.3,0.0
6,Pinar,Sur,16,81.3,56.3,93.8,25.0
7,Solymar,Sur,160,81.9,85.6,95.0,13.8
8,Lagomar,Sur,22,90.9,90.9,90.9,13.6
9,Shangrila,Sur,52,76.9,73.1,92.3,9.6


### **Diagnóstico: Equipamiento por barrio y zona**
1. La "ventaja de piscina" del norte es un efecto La Tahona, con el corte por barrio y zona, La Tahona aparece primera con un contundente 75.7% de piscina, muy por encima de cualquier otra combinación. El resto de los barrios de la zona norte tienen 0% o valores bajos (Lagomar 0% Pinar 0% Shangrilá 0% Solymar 7.1%). Si se excluyera La Tahona del cálculo la zona norte tendría menos piscinas que la zona sur.
2. Carrasco norte cuenta con 40.0% de piscina, muy por encima del resto de los barrios (excluyendo La Tahona). Esto es coherente con lo que ya vimos en la Pregunta 2: Carrasco norte tiene muy pocas viviendas reales (solo 10) así que este porcentaje **hay que leerlo con cautela** (son 4 propiedades con piscina sobre una base chica, no una tendencia robusta de todo el barrio).
3. Con esos dos casos aislados el resto del Dataset muestra que la piscina es un *amenity de nicho* prácticamente ausente en la zona sur y en la mayoría del norte. Los valores de la zona sur van de 7.1% a 25.0% sin ningún barrio destacándose fuertemente.
4. Cochera, parrillero y jardín no muestran un patrón norte/sur consistente. Por ejemplo, Lagomar norte tiene 100% de cochera y jardín, mientras que Pinar norte tiene el parrillero más bajo de toda la tabla (33.3%). Sn variaciones específicas de cada barrio, no un patrón sistemático de zona.
---
### **Pregunta 4. Volatilidad de precio**
Ya sabemos que Carrasco norte tiene la brecha media/mediana más extrema del proyecto (97.4%).

Esta pregunta cuantifica ese mismo fenómeno de forma directa con desvío estándar y coeficiente de variación, para todo el cruce barrio y zona (así identificamos con precisión qué combinaciones son más predecibles en precio y cuáles son más riesgosas de estimar).

In [ ]:
# =============================================================================
# VOLATILIDAD DE PRECIO POR BARRIO Y ZONA
# =============================================================================

# Calculamos, para cada combinación de barrio + zona, el precio promedio,
# el desvío estándar y el coeficiente de variación del precio de venta.
query_volatilidad_barrio_zona = """
SELECT
    ubicacion AS barrio,
    zona,

    -- Cantidad de viviendas construidas en venta consideradas.
    COUNTIF(LOWER(tipo_inmueble) != 'terreno'
            AND LOWER(moneda) = 'u$s'
            AND LOWER(operacion) = 'venta') AS cantidad_propiedades,

    -- Precio promedio de venta de vivienda, en USD.
    ROUND(AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno'
                    AND LOWER(moneda) = 'u$s'
                    AND LOWER(operacion) = 'venta'
               THEN precio END), 0) AS precio_promedio_usd,

    -- Desvío estándar del precio: mide qué tan dispersos están los
    -- valores respecto al promedio dentro de esta combinación específica.
    ROUND(STDDEV_SAMP(CASE WHEN LOWER(tipo_inmueble) != 'terreno'
                            AND LOWER(moneda) = 'u$s'
                            AND LOWER(operacion) = 'venta'
                       THEN precio END), 0) AS desvio_estandar_usd,

    -- Coeficiente de variación (%): desvío estándar como porcentaje del
    -- promedio, para comparar volatilidad entre combinaciones de forma
    -- relativa, sin que el nivel de precio absoluto distorsione la lectura.
    ROUND(
        SAFE_DIVIDE(
            STDDEV_SAMP(CASE WHEN LOWER(tipo_inmueble) != 'terreno'
                              AND LOWER(moneda) = 'u$s'
                              AND LOWER(operacion) = 'venta'
                         THEN precio END),
            AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno'
                      AND LOWER(moneda) = 'u$s'
                      AND LOWER(operacion) = 'venta'
                 THEN precio END)
        ) * 100, 1
    ) AS coeficiente_variacion_pct
FROM
    `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
GROUP BY
    barrio, zona
ORDER BY
    coeficiente_variacion_pct DESC;
"""

# Ejecutamos la consulta y la cargamos en el DataFrame de Pandas.
df_volatilidad_barrio_zona = client.query(query_volatilidad_barrio_zona).to_dataframe()

# Desplegamos el resultado.
df_volatilidad_barrio_zona

,barrio,zona,cantidad_propiedades,precio_promedio_usd,desvio_estandar_usd,coeficiente_variacion_pct
0,Carrasco,Norte,9,296067.0,352181.0,119.0
1,Shangrila,Norte,4,210000.0,126293.0,60.1
2,Lagomar,Sur,16,348563.0,167983.0,48.2
3,Pinar,Norte,7,110286.0,52652.0,47.7
4,Pinar,Sur,11,331455.0,154101.0,46.5
5,Solymar,Sur,125,304898.0,123152.0,40.4
6,Solymar,Norte,58,215981.0,70991.0,32.9
7,Shangrila,Sur,28,315286.0,96335.0,30.6
8,Tahona,Norte,30,650033.0,189584.0,29.2
9,Lagomar,Norte,9,197889.0,32655.0,16.5


### **Diagnóstico: Volatilidad de precio por barrio y zona**
**1. Carrasco norte es la combinación más volátil de todo el proyecto:** coeficiente de variación del 119.0%, esto significa que el desvío estándar (USD 352,181) es mayor que el propio precio promedio (USD 296,067), matemáticamente es un nivel de dispersión extremo, coherente con todo lo que veníamos viendo (67% terreno, brecha media/mediana de 97% y ahora la volatilidad más alta de la tabla). Los cuatro hallazgos de las Preguntas 1, 2 y 4 cuentan la misma historia desde ángulos distintos: Carrasco norte es una combinación con muestra chica y oferta extremadamente heterogénea, casi imposible de resumir con un solo número.

**2. Shangrilá norte ocupa el segundo lugar (60.1%):** también con una muestra muy chica (4 propiedades), un patrón similar al de Carrasco aunque con menos intensidad.

**3. El podio de mayor volatilidad está compuesto casi enteramente por combinaciones de muestra chica:** Carrasco norte: 9, Shangrilá norte: 4, Pinar norte: 7. Reforzando que buena parte de la volatilidad extrema que vemos acá es al menos en parte, un efecto de tamaño de muestra, no necesariamente un mercado "caótico" en la realidad.

**4. Las combinaciones más estables:** Carrasco sur (15.9%) y Lagomar norte (16.5%). Es interesante notar que Carrasco tiene el cruce más volátil (norte) y uno de los más estables (sur) de toda la tabla dentro del mismo barrio (la mejor evidencia de que hablar de Carrasco como un solo mercado sería engañoso; son dos mercados completamente distintos según la zona).

**5. La Tahona tiene una volatilidad moderada (29.2%):** con 30 propiedades, es una muestra bastante más grande que la mayoría de las combinaciones del podio de volatilidad. Confirma que La Tahona es un segmento premium pero predecible.

**6.Solymar sur muestra una volatibilidad considerable (40.4%):** a diferencia de los casos anteriores acá no se puede atribuir a muestra chica, es una dispersión real dentro de un mercado grande y activo.

---
### **Pregunta 5. Premio de precio por proximidad**
Buscamos identificar si estar "próximo a" o "sobre" alguna referencia específica (Costanera, Giannattasio, Car One, Aeropuerto, etc.) se traduce en un precio de venta más alto, controlando por barrio para no confundir el efecto de la proximidad con el efecto de estar en un barrio más caro en general.

In [ ]:
# =============================================================================
# PREMIO DE PRECIO POR PROXIMIDAD (DETALLES)
# =============================================================================

# Calculamos, para cada referencia de proximidad, el precio promedio
# y la mediana de venta de vivienda, junto con la cantidad de casos,
# para ver si alguna proximidad específica se asocia a un precio mayor.
query_precio_detalles = """
SELECT
    detalles,
    zona,

    -- Cantidad de viviendas en venta consideradas para este detalle.
    COUNTIF(LOWER(tipo_inmueble) != 'terreno'
            AND LOWER(moneda) = 'u$s'
            AND LOWER(operacion) = 'venta') AS cantidad_propiedades,

    -- Precio promedio de venta de vivienda, en USD.
    ROUND(AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno'
                    AND LOWER(moneda) = 'u$s'
                    AND LOWER(operacion) = 'venta'
               THEN precio END), 0) AS precio_promedio_usd,

    -- Precio por m2 edificado, calculado a nivel de cada propiedad
    -- individual antes de promediar (misma lógica robusta de siempre).
    ROUND(AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno'
                    AND LOWER(moneda) = 'u$s'
                    AND LOWER(operacion) = 'venta'
                    AND mts2_edificado IS NOT NULL
                    AND mts2_edificado > 0
               THEN SAFE_DIVIDE(precio, mts2_edificado) END), 0) AS precio_x_m2_promedio_usd
FROM
    `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
WHERE
    -- Mismo filtro de categorías de proximidad geográfica que en la
    -- consulta anterior, normalizado con TRIM/LOWER por seguridad.
    LOWER(TRIM(detalles)) IN (
        'próximo a costanera', 'sobre costanera',
        'próximo a giannattasio', 'sobre giannattasio',
        'próximo a car one',
        'próximo a aeropuerto',
        'próximo a interbalnearia', 'sobre interbalnearia',
        'próximo a almenara mall',
        'próximo a tahona',
        'próximo a parque roosevelt',
        'próximo a arroyo'
    )
GROUP BY
    detalles, zona
HAVING
    -- Filtramos categorías con muy pocos casos, ya que un promedio con
    -- 1 o 2 propiedades no es representativo de "premio de proximidad".
    cantidad_propiedades >= 5
ORDER BY
    precio_x_m2_promedio_usd DESC;
"""

# Ejecutamos la consulta y la cargamos en el DataFrame de Pandas.
df_precio_detalles = client.query(query_precio_detalles).to_dataframe()

# Desplegamos el resultado.
df_precio_detalles

,detalles,zona,cantidad_propiedades,precio_promedio_usd,precio_x_m2_promedio_usd
0,Sobre Costanera,Sur,8,326250.0,2865.0
1,Próximo a Almenara Mall,Norte,10,197600.0,2703.0
2,Próximo a Giannattasio,Sur,9,242222.0,2693.0
3,Próximo a Costanera,Sur,43,274981.0,2554.0
4,Próximo a Car One,Norte,13,206146.0,2304.0
5,Próximo a Giannattasio,Norte,9,183222.0,2098.0


### **Diagnóstico: Premio de precio por proximidad**
1. "Sobre Costanera" es la referencia con mayor valor por m2 confirmando el premio de frente costero. Con USD 2,865/m2 las propiedades que están literalmente "sobre" la Costanera (no solo "próximas") pagan el precio más alto de toda la tabla, un premio lógico por ubicación directa sobre la línea de costa.
2. "Próximo a Almenara Mall" (norte) tiene el segundo valor por m2 más alto (USD 2,703) superando incluso a "Próximo a Costanera" (USD 2,554). Esto es contraintuitivo si uno asume que "todo lo del sur vale más", demuestra que la proximidad a un centro comercial específico puede generar un premio de valor comparable al de estar cerca de la costa, incluso estando en la zona norte.
3. Estar sobre vale más que estar próximo en la misma referencia, comparando dentro de Costanera: "Sobre" (USD 2,865/m2) y "Próximo a" (USD 2,554/m2), una diferencia de 12% solo por la precisión de la ubicación. Es un dato accionable: la palabra exacta usada en la publicación ("sobre" vs. "próximo") parece capturar una diferencia real de valor, no solo un matiz de redacción.
4. Giannattasio muestra el mismo premio de zona que ya conocíamos: sur>norte incluso para la misma referencia. "Próximo a Giannattasio" vale USD 2,693/m2 en el sur pero solo USD 2,098/m2 en el norte, una diferencia de 28% para la misma referencia de proximidad. Esto confirma que el efecto zona (sur más caro) persiste incluso controlando por la misma característica de ubicación, reforzando que no es solo la proximidad lo que determina el precio sino también en qué "lado" general del mapa está esa proximidad.
5. "Próximo a Car One" (norte) es la referencia con el valor por m2 más bajo del grupo filtrado (USD 2,304) consistente con el perfil general de la zona norte que ya conocemos (más económica y orientada a desarrollo).

----

## **Cierre del Módulo:**
En este módulo comparamos norte y sur segmentando siempre por barrio para evitar que un caso aislado distorsione la lectura general de una zona. Confirmamos que la brecha de precio responde a diferencias reales de tipología, equipamiento y proximidad geográfica, no a un simple efecto de "lado de la avenida".

## **Próximo Módulo:**
Equipamiento y valor agregado.

---
Análisis realizado en Julio/2026 con fines académicos.
---